In [ ]:
import pandas as pd
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.multioutput import ClassifierChain
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -----------------------------
# 1. Load multilabel dataset
# -----------------------------
df = pd.read_csv("multilabel_dataset.csv")

# Split features and labels
# (Assumes label columns are 0/1 and features are the rest)
label_cols = [col for col in df.columns if df[col].dropna().isin([0,1]).all()]
feature_cols = [col for col in df.columns if col not in label_cols]

X = df[feature_cols]
Y = df[label_cols]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

# -----------------------------
# 2. Stratified split
# -----------------------------
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, test_idx in mskf.split(X, Y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    Y_train, Y_test = Y.iloc[train_idx], Y.iloc[test_idx]
    break

# -----------------------------
# 3. Model
# -----------------------------
base_model = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

model = ClassifierChain(
    base_estimator=base_model,
    order='random',
    random_state=42
)

model.fit(X_train, Y_train)

# -----------------------------
# 4. Prediction
# -----------------------------
probs_list = []

for p in model.predict_proba(X_test):
    probs_list.append(p[:, 1] if len(p.shape) == 2 else p)

Y_pred_prob = np.column_stack(probs_list)

# Fix orientation if needed
if Y_pred_prob.shape[0] == Y.shape[1]:
    Y_pred_prob = Y_pred_prob.T

# -----------------------------
# 5. Thresholding
# -----------------------------
thresholds = Y_train.mean(axis=0).values
Y_pred = (Y_pred_prob >= thresholds).astype(int)

# -----------------------------
# 6. Evaluation
# -----------------------------
print("Subset Accuracy:", accuracy_score(Y_test, Y_pred))

print("\n--- Micro ---")
print("Precision:", precision_score(Y_test, Y_pred, average='micro', zero_division=0))
print("Recall:", recall_score(Y_test, Y_pred, average='micro', zero_division=0))
print("F1:", f1_score(Y_test, Y_pred, average='micro', zero_division=0))

print("\n--- Macro ---")
print("Precision:", precision_score(Y_test, Y_pred, average='macro', zero_division=0))
print("Recall:", recall_score(Y_test, Y_pred, average='macro', zero_division=0))
print("F1:", f1_score(Y_test, Y_pred, average='macro', zero_division=0))

X shape: (46238, 126)
Y shape: (46238, 7)
